In [ ]:
%%bash
cd /content
rm -rf LatentSync
echo " Cloning LatentSync Repository..."
git clone https://github.com/bytedance/LatentSync.git
cd LatentSync

echo " Applying precise version fixes..."
sed -i 's/mediapipe==0.10.11/mediapipe==0.10.14/g' requirements.txt

apt-get update -q
apt-get install -y -q ffmpeg

echo " Installing Dependencies..."
pip install -q -r requirements.txt
pip install -q huggingface_hub decord kornia insightface ffmpeg-python DeepCache
pip install -q --upgrade protobuf accelerate peft
pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio diffusers transformers

echo " Fixing PyTorch & Torchvision Mismatch..."
pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121

echo "✅ Environment Setup Complete!"

 Cloning LatentSync Repository...
 Applying precise version fixes...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,812 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,085 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu 

Cloning into 'LatentSync'...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
python-fasthtml 0.14.3 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.3.0 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.3.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nest_asyncio

In [ ]:
%%writefile server_stage4_latentsync.py
import os
import subprocess
import requests
import uvicorn
import gc
import asyncio
import nest_asyncio

from fastapi import FastAPI
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok

# Setup paths
DRIVE_MODELS_CACHE = '/content/drive/MyDrive/Video_Translation_Project/Models_Cache/LatentSync_Checkpoints'
LOCAL_CKPT_DIR = '/content/LatentSync/checkpoints'

print("[INFO] Linking models from Google Drive to LatentSync...")
if os.path.exists(LOCAL_CKPT_DIR):
    os.system(f"rm -rf {LOCAL_CKPT_DIR}")
os.symlink(DRIVE_MODELS_CACHE, LOCAL_CKPT_DIR)
print("[INFO] Models linked successfully.")

# Reset the script to clean state
os.system("cd /content/LatentSync && git checkout -- scripts/inference.py")

# OPTIMIZATION 1: Force FP16 and Low Memory target sizes in the YAML Config
CONFIG_PATH = "/content/LatentSync/configs/unet/stage2_efficient.yaml"
if os.path.exists(CONFIG_PATH):
    print("[INFO] Patching LatentSync config for FP16 and optimized VRAM usage...")
    with open(CONFIG_PATH, "r") as f:
        config_data = f.read()

    # Lower resolution targets internally to save memory
    config_data = config_data.replace("sample_size: 512", "sample_size: 256")
    config_data = config_data.replace("sample_size: 768", "sample_size: 256")
    # Force float16 if it's explicitly set in config
    config_data = config_data.replace("fp32", "fp16")
    config_data = config_data.replace("float32", "float16")

    with open(CONFIG_PATH, "w") as f:
        f.write(config_data)

# OPTIMIZATION 2: Hard-patch the Inference Script to load models in FP16
INFERENCE_SCRIPT = "/content/LatentSync/scripts/inference.py"
if os.path.exists(INFERENCE_SCRIPT):
    print("[INFO] Injecting FP16 dtype override into inference.py...")
    with open(INFERENCE_SCRIPT, "r") as f:
        inf_code = f.read()

    # Replace default torch float types with half precision
    inf_code = inf_code.replace("torch.float32", "torch.float16")
    inf_code = inf_code.replace("weight_dtype=torch.float32", "weight_dtype=torch.float16")

    with open(INFERENCE_SCRIPT, "w") as f:
        f.write(inf_code)

class VideoRequest(BaseModel):
    silent_video_url: str
    arabic_audio_url: str

app = FastAPI(title="Stage 4: LatentSync Rendering Engine (FP16 Edition)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/process_stage3_from_urls")
async def process_stage3_from_urls(payload: VideoRequest):
    gc.collect()

    vid_original = "/content/silent_original.mp4"
    vid_in = "/content/silent_in.mp4"
    aud_in = "/content/arabic_in.wav"
    vid_out = "/content/final_output_lipsynced.mp4"

    print("[INFO] Downloading files securely...")
    headers = {'ngrok-skip-browser-warning': 'true'}

    with open(vid_original, 'wb') as f:
        f.write(requests.get(payload.silent_video_url, headers=headers).content)
    with open(aud_in, 'wb') as f:
        f.write(requests.get(payload.arabic_audio_url, headers=headers).content)

    print("[INFO] Downscaling video to 480p (Perfect balance for Face Detection & VRAM)...")
    # 480p is large enough to avoid 'Face not detected' errors, but small enough to run smoothly in FP16
    ffmpeg_cmd = [
        "ffmpeg", "-y", "-i", vid_original,
        "-vf", "scale=-2:480",
        "-c:a", "copy", vid_in
    ]
    subprocess.run(ffmpeg_cmd, capture_output=True)

    print("[INFO] Starting LatentSync render in FP16 Mode...")
    CKPT_PATH = "/content/LatentSync/checkpoints/latentsync_unet.pt"

    # Aggressive VRAM fragmentation prevention
    custom_env = os.environ.copy()
    custom_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:32"

    cmd = [
        "python", "-m", "scripts.inference",
        "--unet_config_path", CONFIG_PATH,
        "--inference_ckpt_path", CKPT_PATH,
        "--video_path", vid_in,
        "--audio_path", aud_in,
        "--video_out_path", vid_out,
        "--inference_steps", "20",
        "--guidance_scale", "1.5",
        "--seed", "42"
    ]

    process = subprocess.run(
        cmd,
        cwd="/content/LatentSync",
        env=custom_env,
        capture_output=True,
        text=True
    )

    if process.returncode == 0:
        print("[INFO] Stage 4 Complete! Video rendered.")
        return {"status": "Success", "message": "Video rendered successfully."}

    print(f"[ERROR] LatentSync failed:\n{process.stderr}")
    return {"status": "Error", "error": process.stderr}

@app.get("/get_final_video")
def get_final_video():
    return FileResponse("/content/final_output_lipsynced.mp4", media_type="video/mp4")

if __name__ == "__main__":
    nest_asyncio.apply()
    os.system("pkill -f ngrok")
    ngrok.set_auth_token("3BoNqp4E41yckxXld1dTRHbkMT6_7nPrp96pKbxSQKrie4teq")
    public_url = ngrok.connect(8003).public_url
    print(f"\n🚀 STAGE 4 MICROSERVICE IS LIVE AT: {public_url}\n")

    uvicorn.run(app, host="0.0.0.0", port=8003)

Overwriting server_stage4_latentsync.py


In [ ]:
!cat /content/LatentSync/configs/unet/stage2_efficient.yaml

data:
  syncnet_config_path: configs/syncnet/syncnet_16_pixel_attn.yaml
  train_output_dir: debug/unet
  train_fileslist: /mnt/bn/maliva-gen-ai-v2/chunyu.li/fileslist/data_v10_core.txt
  train_data_dir: ""
  audio_embeds_cache_dir: /mnt/bn/maliva-gen-ai-v2/chunyu.li/audio_cache/embeds
  audio_mel_cache_dir: /mnt/bn/maliva-gen-ai-v2/chunyu.li/audio_cache/mel

  val_video_path: assets/demo1_video.mp4
  val_audio_path: assets/demo1_audio.wav
  batch_size: 1 # 4
  num_workers: 12 # 12
  num_frames: 16
  resolution: 256
  mask_image_path: latentsync/utils/mask.png
  audio_sample_rate: 16000
  video_fps: 25
  audio_feat_length: [2, 2]

ckpt:
  resume_ckpt_path: checkpoints/latentsync_unet.pt
  save_ckpt_steps: 10000

run:
  pixel_space_supervise: true
  use_syncnet: true
  sync_loss_weight: 0.05
  perceptual_loss_weight: 0.1 # 0.1
  recon_loss_weight: 1 # 1
  guidance_scale: 1.5 # [1.0 - 3.0]
  trepa_loss_weight: 0
  inference_steps: 20
  trainable_modules:
    - motion_modules.
    - attn2.

In [ ]:
!python server_stage4_latentsync.py

[INFO] Linking models from Google Drive to LatentSync...
[INFO] Models linked successfully.
[INFO] Patching LatentSync config for FP16 and optimized VRAM usage...
[INFO] Injecting FP16 dtype override into inference.py...

🚀 STAGE 4 MICROSERVICE IS LIVE AT: https://unjudicial-consciencelessly-isela.ngrok-free.dev

INFO:     Started server process [3929]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8003 (Press CTRL+C to quit)
INFO:     197.63.240.96:0 - "OPTIONS /process_stage3_from_urls HTTP/1.1" 200 OK
[INFO] Downloading files securely...
[INFO] Downscaling video to 480p (Perfect balance for Face Detection & VRAM)...
[INFO] Starting LatentSync render in FP16 Mode...
[INFO] Stage 4 Complete! Video rendered.
INFO:     197.63.240.96:0 - "POST /process_stage3_from_urls HTTP/1.1" 200 OK
